In [1]:
import sys
sys.path.append('../../../../')

In [2]:
from CADETProcess.metric_space import Metric, MetricSpace

metric_space = MetricSpace()
metric_space.add_metric(Metric('volume'))

Metric(name='volume')

In [3]:
import math
from dataclasses import dataclass

from CADETProcess.parameter_space import ParameterSpace, RangedParameter
from CADETProcess.evaluation_pipeline import EvaluationPipeline

@dataclass
class Column:
    length: float = 0.5
    diameter: float = 0.05

column = Column()
space = ParameterSpace()
space.add_evaluation_object(column)
space.add_parameter(RangedParameter('length', float, lb=0.1, ub=1.0), path='length')
space.add_parameter(RangedParameter('diameter', float, lb=0.01, ub=0.2), path='diameter')

pipeline = EvaluationPipeline(space)
pipeline.add_evaluator(
    lambda col: col.length * (col.diameter / 2) ** 2 * math.pi,
    output_name='volume',
)

In [4]:
from CADETProcess.problem import Problem

problem = Problem(
    parameter_space=space, metric_space=metric_space, backend=pipeline, name='column_volume',
)
problem.evaluate({'length': 0.5, 'diameter': 0.05})

{'volume': array(0.00098175)}

In [5]:
from CADETProcess.parameter_space import LatinHypercubeSampler

samples = LatinHypercubeSampler().sample(space, 8, seed=0)
results = problem.evaluate_batch(samples)
results[:2]

[{'volume': array(0.00365188)}, {'volume': array(0.00511121)}]

In [6]:
from CADETProcess.optimization import Population

records = [{'X': sample, 'metrics': result} for sample, result in zip(samples, results)]
population = Population.from_records(records, metric_space=metric_space, parameter_space=space)
population.X['length'][:5]

[INFO 08-12 16:21:55] ax.storage.sqa_store.with_db_settings_base: Ax SQL storage initialized with SQLAlchemy 2.0.52


array([0.47834181, 0.99539048, 0.6835071 , 0.59425348, 0.15134219])

In [7]:
population[0].X

{'length': np.float64(0.47834181017633637),
 'diameter': np.float64(0.09859256554810808)}

In [8]:
population[0].metrics

{'volume': np.float64(0.0036518805169675414)}

In [9]:
one = Population.from_sample(
    samples[0], results[0], metric_space=metric_space, parameter_space=space,
)
one[0].metrics

{'volume': np.float64(0.0036518805169675414)}